# Hidden Markov Model
Hidden Markov Models (HMMs) contain hidden states that we are trying to infer from observed data. This is useful in bioinformatics, because the observed data is what we can directly measure, like sequenced DNA. On the other hand, the hidden states represent the underlying biological context we are trying to uncover or infer. We will use the Viterbi algorithm to find the most likely sequence of hidden states given a sequence of observations.

The Baum-Welch algorithm represents an Expectation-Maximization approach to learning HMM parameters, offering several advantages over manual parameter setting:

Key Features

* Unsupervised Learning: Estimates model parameters without labeled training data
* Maximum Likelihood: Finds parameters that maximize observation probability
* Iterative Refinement: Progressively improves parameter estimates
* Convergence Guarantees: Always reaches a local optimum of the likelihood function

*
* Algorithm Structure

The Baum-Welch algorithm involves these key steps:

**Initialization:**
* Start with initial guesses for transition, emission, and initial probabilities
* Set up convergence criteria and pseudocounts to prevent zero probabilities

**Expectation Step (E):**
* Run Forward-Backward algorithm on training sequences
* Calculate expected counts for transitions and emissions
* Compute posterior probabilities for each state at each position
* These expected counts represent how often each transition and emission is used to generate the observed sequences

**Maximization Step (M):**
* Update model parameters based on expected counts
* Re-estimate initial, transition, and emission probabilities using the expected counts
* Normalize to ensure valid probability distributions
* Scale values to prevent numerical underflow

**Iteration and Convergence:**
* Repeat the E and M steps until convergence criteria are met
* Monitor likelihood improvement between iterations
* Handle multiple observation sequences appropriately


In [21]:
import numpy as np
import math
from pprint import pprint
import hmm_utils

In [2]:
class BaumWelch():
    def __init__(self, num_states, seed, pseudocount=1e-14, num_sweeps=10000, convergence_threshold = 1e-6):
        self.num_states = num_states
        self.states = [f"S{i}" for i in range(num_states)]
        self.seed = seed
        self.nucleotide_map = {"A": 0, "C": 1, "G": 2, "T": 3}
        self.num_sweeps = num_sweeps
        self.pseudocount = pseudocount
        self.initial_probs, self.transition_probs, self.emission_probs = self.bw_initialization()
        self.emission_counts = np.full((self.num_states, 4), -np.inf)
        self.transition_counts = np.full((self.num_states, self.num_states), -np.inf)
        self.initial_counts = np.full((self.num_states,), -np.inf)
        self.total_seq_prob = -np.inf
        self.convergence_threshold = convergence_threshold



    def bw_initialization(self):
        """
        Initialize the initial, transition, and emission probability matrices using dirichlet to produce a series of distributions that sum to 1 across each row of the matrix. All values are transformed into log space to be ready for calculations. 
        
        return:
            initial_probs: np.ndarray - initial probabilities, shape (1 x num_states) in log space
            transition_probs: np.ndarray - transition probabilities, shape (num_states x num_states) in log space
            emission_probs: np.ndarray - emission probabilities, shape (num_states x nucleotide_map.keys()) in log space
        """

        # initialize random seed
        rand = np.random.default_rng(self.seed)

        # initial probs
        initial_probs = rand.dirichlet(np.ones(self.num_states))
        initial_probs += self.pseudocount
        initial_probs = np.log(initial_probs)

        # transmission probs
        transition_probs = rand.dirichlet(np.ones(self.num_states), size=self.num_states)
        transition_probs += self.pseudocount
        transition_probs = np.log(transition_probs)

        # emission probs
        emission_probs = rand.dirichlet(np.ones(len(self.nucleotide_map.keys())), size=self.num_states)
        emission_probs += self.pseudocount
        emission_probs = np.log(emission_probs)

        return initial_probs, transition_probs, emission_probs

    def forward_matrix(self, observations):
        """
        The algorithm computes the forward probability matrix in log-space. Each entry represents the probability of having emitted the observed symbols up to the particular position and state.

        params:
            observations: list of observed sequences
        return:
            prob_matrix: np.ndarray - Forward probability matrix of shape (num_states x len(observations)) in log space
        """
        
        prob_matrix = np.full((self.num_states, len(observations)), -np.inf)

        for i, observation in enumerate(observations):
            if i == 0:
                prob_matrix[:, i] = self.initial_probs + self.emission_probs[:, self.nucleotide_map[observation]]
            else:
                prev = prob_matrix[:, i-1][:, np.newaxis] + self.transition_probs
                prob_matrix[:, i] = np.logaddexp.reduce(prev, axis = 0) + self.emission_probs[:, self.nucleotide_map[observation]]

        return prob_matrix

    def backward_matrix(self, observations):
        """
        Computes the backward probability matrix by reversing the sequence and then re-reversing the matrix to return. Each entry represents the probability of having emitted the observed symbols from the current positon to the end of the sequence given a specific state.

        params:
            observations: list of observed sequences
        return:
            prob_matrix: np.ndarray - Backward probability matrix of shape (num_states x len(observations)) in log space
        """
        
        reversed_obs = observations[::-1]
        prob_matrix = np.zeros((self.num_states, len(observations)), dtype=float)

        for i, observation in enumerate(reversed_obs):
            #for j, state in enumerate(self.states):
            if i == 0:
                continue
            else:
                prev = prob_matrix[:, i-1][np.newaxis, :] + self.transition_probs
                prob_matrix[:, i] = np.logaddexp.reduce(prev, axis = 1) + self.emission_probs[:, self.nucleotide_map[reversed_obs[i-1]]]

        return np.fliplr(prob_matrix)

    def sequence_probability(self, prob_matrix):
        """
        Computes the total probability of the observation sequence from a probability matrix.
        params:
            prob_matrix: np.ndarray - Probability matrix of shape (num_states x len(observations))
        return:
            float - Log probability of the observation sequence
        """
        
        return np.logaddexp.reduce(prob_matrix[:, -1])

    def forward_backward(self, observations):
        """
        Computes the forward-backward posterior matrix by combining the forward and backward matrices, normalized by the total probability of the observation sequence.
        params:
            observations: list of observed sequences
        return:
            forward_matrix: np.ndarray - Forward probability matrix of shape (num_states x len(observations)) in log space
            backward_matrix: np.ndarray - Backward probability matrix of shape (num_states x len(observations)) in log space
            posterior_matrix: np.ndarray - posterior probabilities matrix of shape (num_states x len(observations)) in log space
        """
        
        self.posterior_matrix = np.zeros((self.num_states, len(observations)), dtype=float)

        forward_matrix = self.forward_matrix(observations)
        backward_matrix = self.backward_matrix(observations)

        final_col_prob = self.sequence_probability(forward_matrix)

        for i in range(len(observations)):
            for j in range(self.num_states):
                self.posterior_matrix[j][i] = forward_matrix[j][i] + backward_matrix[j][i] - final_col_prob

        return forward_matrix, backward_matrix, self.posterior_matrix

    def expectation(self, observations):
        """
        Executes Expectation step of Baum-Welch, looping over all observations and accumulating initial, transition, emission counts, and total sequence probabilities.
        params:
            observations: list of observed sequences
        """

        for obs in observations:
            # Create intermediate matrices
            forward, backward, posterior = self.forward_backward(obs)
            transition_posterior = self.transition_posterior_matrix(forward, backward, obs)

            # Accumulation
            self.accumulate_emission_counts(posterior, obs)
            self.accumulate_transition_counts(transition_posterior)
            self.accumulate_initial_counts(posterior)
            self.accumulate_total_seq_prob(forward)

    def transition_posterior_matrix(self, forward, backward, observation):
        """
        Calculates transition posterior matrix for Expectation step of Baum-Welch.
        params:
            forward: np.ndarray - Forward probability matrix of shape (num_states x len(observations)) in log space
            backward: np.ndarray - Backward probability matrix of shape (num_states x len(observations)) in log space
            observation: str - observed sequence
        return:
            trans_posterior: np.ndarray - posterior transition values, 3d shape (num_states x len(observation) - 1 x num_states)
        """
        # Create a 3D matrix filled with -inf (log-space)
        final_col_prob = self.sequence_probability(forward)
        trans_posterior = np.full((self.num_states, len(observation)-1, self.num_states), -np.inf)

        for n in range(len(observation)-1):
            # Vectorization
           trans_posterior[:, n] = (forward[:, n][:, np.newaxis] + 
            self.transition_probs + 
            self.emission_probs[:, self.nucleotide_map[observation[n+1]]] + 
            backward[:, n+1][np.newaxis, :] - 
            final_col_prob)

        # Collapse transition posterior into a 2d array
        trans_posterior = np.logaddexp.reduce(trans_posterior, axis =1)

        return trans_posterior

    def accumulate_emission_counts(self, posterior, sequence):
        """
        Calculates emission counts matrix for Expectation step of Baum-Welch.
        params:
            posterior: np.ndarray - posterior matrix
            sequence: str - observed sequence
        """        
        for i, pos in enumerate(sequence):
            pos_index = self.nucleotide_map[pos]
            self.emission_counts[:, pos_index] = np.logaddexp(self.emission_counts[:, pos_index], posterior[:, i])

    def accumulate_transition_counts(self, trans_posterior):
        """
        Calculates transition counts matrix for Expectation step of Baum-Welch.
        params:
            trans_posterior: np.ndarray - transmission posterior matrix
        """
        self.transition_counts = np.logaddexp(self.transition_counts, trans_posterior)

    def accumulate_initial_counts(self, posterior):
        """
        Calculates initial counts matrix for Expectation step of Baum-Welch.
        params:
            posterior: np.ndarray - posterior matrix
        """        
        self.initial_counts = np.logaddexp(self.initial_counts, posterior[:, 0])
    
    def accumulate_total_seq_prob(self, forward):
        """
        Calculates total sequence probabilities for Expectation step of Baum-Welch.
        params:
            forward: np.ndarray - forward posterior matrix
        """
        self.total_seq_prob = np.logaddexp(self.total_seq_prob, self.sequence_probability(forward))

    def normalization(self):
        """
        Normalizes initial, emission, transmission matrices for Expectation step of Baum-Welch.
        return:
            normalized_initial_counts: np.ndarray - normalized initial counts matrix
            normalized_emission_counts: np.ndarray - normalized emission counts matrix
            normalized_transition_counts: np.ndarray - normalized transition counts matrix
        """
        # Normalize initial counts by total probability of accumulated sequences
        total_initial = np.logaddexp.reduce(self.initial_counts)
        normalized_initial_counts = self.initial_counts - total_initial

        # Normalize emission counts
        posterior_probs_sum_by_states = np.logaddexp.reduce(self.emission_counts, axis=1)
        normalized_emission_counts = np.logaddexp(self.emission_counts, np.log(self.pseudocount)) - posterior_probs_sum_by_states[:, np.newaxis]

        # Normalize transition counts
        trans_posterior_probs_sum_by_states = np.logaddexp.reduce(self.transition_counts, axis=(1), keepdims=True)
        normalized_transition_counts = np.logaddexp(self.transition_counts, np.log(self.pseudocount)) - trans_posterior_probs_sum_by_states

        return normalized_initial_counts, normalized_emission_counts, normalized_transition_counts

    def maximization(self):
        """
        Executes Maximization step of Baum-Welch. Normalized counts are stored in probability matrices.
        """
        normalized_initial_counts, normalized_emission_counts, normalized_transition_counts = self.normalization()

        # Build the new emission probability
        self.emission_probs = normalized_emission_counts

        # Build the new transitions probability
        self.transition_probs = normalized_transition_counts

        # Build the new initial probability
        self.initial_probs = normalized_initial_counts

    def baum_welch(self, observations):
        """
        Driver for Baum-Welch algorithm. 
        params:
            observations: - list of observed sequences
        returns:
            print statement regarding convergence status
        """
        
        prev_total_seq_prob = None
        # only try for num_sweeps iterations
        for i in range(self.num_sweeps):
            self.expectation(observations)
            self.maximization()

            # Only check every so often
            check_interval = 20
            if (i % check_interval) == 0:                
                print(f"\rCheck total seq prob: Previous: {prev_total_seq_prob} Current: {self.total_seq_prob}", end="", flush=True)
                if prev_total_seq_prob is not None:
                    convergence_check = math.isclose(self.total_seq_prob, prev_total_seq_prob, abs_tol=self.convergence_threshold)
                    if convergence_check:
                        return f"Convergence reached at {i+1} number of sweeps."

                prev_total_seq_prob = self.total_seq_prob

            # Reset counts matrices for next iteration
            self.emission_counts = np.full((self.num_states, 4), -np.inf)
            self.transition_counts = np.full((self.num_states, self.num_states), -np.inf)
            self.initial_counts = np.full((self.num_states,), -np.inf)
            self.total_seq_prob = -np.inf
        # if we got here we didn't converge
        return f"Convergence not reached at {i+1} number of sweeps."

    def model_probs_as_dicts(self):
        """
        Transforms probability matrices into dictionaries for use with previously developed Viterbi and Forward-Backward algorithms.
        return:
        initial_probs_dict: - dict of initial probabilities
        transition_probs_dict: - dict of transition probabilities
        emission_probs_dict: - dict of emission probabilities
        """
        
        initial_probs_dict = {}
        for i, state in enumerate(self.states):
            initial_probs_dict[state] = np.exp(self.initial_probs[i])
        print("Initial Probs:")
        pprint(initial_probs_dict)

        transition_probs_dict = {}
        for i, s1 in enumerate(self.states):
            transition_probs_dict[s1] = {}
            for j, s2 in enumerate(self.states):
                transition_probs_dict[s1][s2] = np.exp(self.transition_probs[i][j])
        print("Transition Probs:")
        pprint(transition_probs_dict)

        emission_probs_dict = {}
        reverse_nucleotide_map = {v: k for k, v in self.nucleotide_map.items()}
        for i, state in enumerate(self.states):
            emission_probs_dict[state] = {}
            for j, symbol in reverse_nucleotide_map.items():
                emission_probs_dict[state][symbol] = np.exp(self.emission_probs[i][j])
        print("Emission Probs:")
        pprint(emission_probs_dict)

        return initial_probs_dict, transition_probs_dict, emission_probs_dict


In [16]:
# Build training and testing sequences using defined probabilities to create pretend "observed data"
# This generates two sets of sequences using the same model: one for training using Baum-Welch and the other to be used for checking the validity of the model.

# States
states = ["GC", "BG"]

# Initial probabilities
initial_probs = {"GC": 0.3, "BG": 0.7}

# Transition probabilities
transition_probs = {
    "GC": {"GC": 0.8, "BG": 0.2},
    "BG": {"GC": 0.1, "BG": 0.9}
}

# Emission probabilities
emission_probs = {
    "GC": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "BG": {"A": 0.4, "C": 0.1, "G": 0.1, "T": 0.4}
}


def generate_sequence(length=30, seed=None):
    rng = np.random.default_rng(seed)
    nucleotides = ["A", "C", "G", "T"]

    state = rng.choice(states, p=[initial_probs["GC"], initial_probs["BG"]])
    seq = []
    hidden = []
    for _ in range(length):
        probs = [emission_probs[state][n] for n in nucleotides]
        nuc = rng.choice(nucleotides, p=probs)
        seq.append(nuc)
        hidden.append(state)
        trans = transition_probs[state]
        state = rng.choice(states, p=[trans["GC"], trans["BG"]])
    return "".join(seq), hidden


# these sequences will be used for training
sequences_train = []
true_states = []
for i in range(30):
    seq, states_seq = generate_sequence(length=30)
    sequences_train.append(seq)
    true_states.append(states_seq)

# these sequences are used for validating the model using the training data
sequences_test = []
true_states = []
for i in range(30):
    seq, states_seq = generate_sequence(length=30)
    sequences_test.append(seq)
    true_states.append(states_seq)


In [17]:
# Initialize the Baum-Welch model
bw_model = BaumWelch(num_states=2, seed=42, num_sweeps=150000, convergence_threshold=1e-9)

# Train the model using the training sequences
bw_model.baum_welch(sequences_train)


Check total seq prob: Previous: -33.76089719289922 Current: -33.7608971622337726

'Convergence reached at 461 number of sweeps.'

In [20]:
# Run Viterbi on the test sequences using the probabilities generated by Baum-Welch
viterbi = hmm_utils.HiddenMarkovModel(*bw_model.model_probs_as_dicts())

# one example
test_seq = sequences_test[1]
print("")
print(test_seq)
print("Optimal path:")
print(viterbi.viterbi_algorithm(test_seq))

# Ideally, you will see probabilities below that are close to those used to generate the "observed" data we trained on.
# Remember that the algorithm does not know which state is which.

Initial Probs:
{'S0': 0.7847128479913604, 'S1': 0.21528715200863968}
Transition Probs:
{'S0': {'S0': 0.9101245079437855, 'S1': 0.08987549205621474},
 'S1': {'S0': 0.09843650426867667, 'S1': 0.9015634957313229}}
Emission Probs:
{'S0': {'A': 0.3836430760209901,
        'C': 0.07725182991226727,
        'G': 0.09663768131977889,
        'T': 0.4424674127469635},
 'S1': {'A': 0.14954666072839437,
        'C': 0.340353773086289,
        'G': 0.328075625280249,
        'T': 0.1820239409050678}}

GAGGAGGCATATATGACGCGACTGAAGTAT
Optimal path:
[['S1', 'S1', 'S1', 'S1', 'S1', 'S1', 'S1', 'S1', 'S0', 'S0', 'S0', 'S0', 'S0', 'S0', 'S1', 'S1', 'S1', 'S1', 'S1', 'S1', 'S1', 'S1', 'S1', 'S1', 'S0', 'S0', 'S0', 'S0', 'S0', 'S0']]
